## 🕸️ Web Scraping Cheatsheet (HTML + BeautifulSoup)

---

### 1. 🧠 What is Web Scraping?
**Web scraping** is the automated process of extracting structured information from websites. 
It allows efficient gathering of data e.g. job listings, product data, news articles and more without manual copy and pasting

---

### 2. 🧭 What Are We Trying to Do?
Web scraping usually involves: 
- Loading a webpage (HTML Source)
- Parsing the HTML using a parser such as BeautifulSoup
- Locating specific elements using tag names, classes, or attributes 
- Extracting data: text, links, images, prices etc
- Saving to a format like CSV, JSON, or database 

Example:
> Extract all product names, prices, ratings, and links from an e-commerce site.
---

---

### 3. 🧱 Anatomy of a Typical Web Page (HTML)

A webpage is structured as a **tree of nested HTML tags**, often referred to as the **DOM** (Document Object Model).  
Each tag is an **element**, and can contain **attributes** like `class`, `id`, `href`, etc.

---

#### 🔍 Realistic Product Listing Example

```html
<div class="product-card">
  <div class="product-header">
    <h2 class="product-name">iPhone 14 Pro</h2>
    <span class="badge">Bestseller</span>
  </div>

  <div class="product-meta">
    <span class="price">$999</span>
    <span class="rating">⭐ 4.7</span>
    <span class="reviews">(324 reviews)</span>
  </div>

  <div class="product-actions">
    <a href="/product/iphone-14-pro" class="buy-link">Buy Now</a>
    <img src="/images/iphone14.jpg" alt="iPhone 14 Pro">
  </div>
</div>
```

---

### 🧠 Key Takeaways

#### ✅ What is an "Element"?

An **element** is any individual piece of HTML on the page — typically represented by a tag like `<div>`, `<span>`, `<a>`, or `<img>`.

```html
<span class="price">$999</span>   <!-- This is one element -->
```

---

#### ✅ What Are "Labels"?

"Labels" are small pieces of content that **identify or describe** something — like price, rating, or tags. They’re often wrapped in `<span>`, `<p>`, or `<small>` tags.

```html
<span class="price">$999</span>
<span class="badge">Bestseller</span>
```

---

#### ✅ What Are "Actions" and "Links"?

These are **interactive elements**, like buttons or hyperlinks:

- `<a>` for navigation (links to product pages)
- `<button>` for actions (e.g. “Add to Cart”)

```html
<a href="/product/iphone-14-pro" class="buy-link">Buy Now</a>
```

You can extract:
- Link text → `'Buy Now'`
- Link URL → `'/product/iphone-14-pro'`

---

#### ✅ What is a CSS Class?

A **CSS class** is an attribute used to:
- Style an element using CSS
- Group similar elements together semantically

**Example:**
```html
<div class="product-card">...</div>
```

You’ll commonly scrape based on class:

```python
soup.find('div', class_='product-card')
```

Note:
- Classes are not unique (many elements can share a class)
- Multiple classes can appear in one tag:  
  `<div class="product-card featured">`

---

### 🧩 Where JavaScript Comes In (And When You Need Selenium)

Many modern websites use **JavaScript to load content dynamically**, after the initial HTML loads.  
When this happens, `requests.get()` will return **incomplete HTML**, missing product names, prices, or table rows.

#### ✅ Examples of JavaScript-Rendered Content

| Feature | Description |
|---------|-------------|
| Infinite scroll | Loads more items as you scroll |
| Tabs or filters | Loads new data without reloading the page |
| Live charts, ratings, countdowns | Rendered using JavaScript and AJAX |
| Delayed content | Appears a few seconds after page load |

---

#### 🚫 Problem with `requests` + `BeautifulSoup`

```python
response = requests.get("https://some-js-heavy-site.com")
print(response.text)  # Page structure is there, but product data is missing!
```

---

### ⚙️ Solution: Use Selenium for Dynamic Pages

**Selenium** is a Python library that controls a **real browser**, like Chrome or Firefox.  
It loads the full page, runs JavaScript, and lets you extract content once everything is rendered.

```python
from selenium import webdriver
from bs4 import BeautifulSoup

driver = webdriver.Chrome()
driver.get("https://some-js-heavy-site.com")
html = driver.page_source

soup = BeautifulSoup(html, 'html.parser')
driver.quit()
```

> 🧠 Selenium opens a real browser window unless used in **headless mode**, which hides the UI.

---

### 🤖 When to Use: `Selenium` vs `requests`

| Feature | `requests` + `BeautifulSoup` | `Selenium` |
|--------|-----------------------------|------------|
| Static pages (HTML loads all data) | ✅ Fast and lightweight | 😐 Overkill |
| Dynamic content (JS/AJAX) | ❌ Can't access data | ✅ Full page rendered |
| Need to click, scroll, log in | ❌ Not possible | ✅ Automate clicks & forms |
| Need speed / large scale | ✅ Efficient | ❌ Slower and resource-intensive |
| Want to simulate real browser | ❌ No JavaScript | ✅ Real browser behavior |

---

### 🚀 Tip: How to Decide

Ask yourself:

- Can you find the data using **View Page Source**?
  - ✅ Yes → use `requests`
  - ❌ No → use **Selenium** or check browser's **Network tab** for an API

- Is the site really slow or protected?
  - Consider headless browsers or **Playwright** (modern alternative to Selenium)

---

### 4. 🔎 Common HTML Tags & Attributes

To scrape data from a webpage, it’s crucial to understand how HTML is structured.

---

#### 🧱 What is an HTML Tag?

An **HTML tag** is a building block of a webpage. Tags define elements on a page, such as text blocks, links, images, or layout containers.

**Syntax:**
```html
<tagname>Content</tagname>
```

Example:
```html
<h1>Welcome</h1>  <!-- h1 is a tag that renders a large heading -->
```

---

#### 🧩 What Are Self-Closing Tags?

A **self-closing tag** is an HTML tag that doesn’t wrap around content — it stands alone and doesn't need a closing tag.

These tags usually represent **empty elements** like images or inputs, which don’t contain nested content.

**Syntax:**
```html
<tagname />
```

**Common Examples:**

| Tag | Purpose | Example |
|-----|---------|---------|
| `<img />` | Embeds an image | `<img src="logo.png" alt="Site logo" />` |
| `<input />` | Input field in a form | `<input type="text" name="email" />` |
| `<br />` | Line break (like pressing "Enter") | `<br />` |
| `<hr />` | Horizontal rule (divider line) | `<hr />` |
| `<meta />` | Metadata in `<head>` | `<meta charset="UTF-8" />` |

> 💡 In modern HTML5, the trailing slash (`/`) is optional:
```html
<img src="logo.png" alt="Logo">  <!-- also valid -->
```

When scraping:
- You can still access their **attributes** (`src`, `name`, `value`, etc.)
- But they **won’t have inner text** — so don’t try to use `.text` on them!

---

#### 🔗 What is an Attribute?

Attributes provide **additional information** about an element.  
They appear **inside the opening tag**, and consist of a `key="value"` pair.

**Example:**
```html
<a href="https://example.com" class="button">Click Me</a>
```

In this example:
- `href` tells where the link goes
- `class` can be used for styling or targeting with CSS or scraping tools

---

#### 🧱 Common HTML Tags (for Scraping)

| Tag | Description |
|-----|-------------|
| `<div>` | A generic block-level container; often used to group content |
| `<span>` | An inline container, typically used for small pieces of content like price or date |
| `<a>` | Anchor tag — used to create hyperlinks (`href` is the URL) |
| `<img>` | Displays an image; the image source is in the `src` attribute |
| `<ul>`, `<li>` | Lists (`ul` is unordered list, `li` is list item) |
| `<table>`, `<tr>`, `<td>` | Table layout: row (`tr`), cell (`td`) |
| `<form>`, `<input>` | Forms for user input, search boxes, logins, etc. |
| `<button>` | Clickable button element |
| `<h1>` to `<h6>` | Heading levels (h1 is biggest, h6 is smallest) |
| `<p>` | Paragraph of text |

---

#### ⚙️ Common HTML Attributes (for Targeting or Extraction)

| Attribute | Purpose | Example |
|-----------|---------|---------|
| `class` | Groups elements for CSS styling or identification | `class="price"` |
| `id` | Unique element identifier on the page | `id="main-header"` |
| `href` | Link destination (used in `<a>`) | `href="/about"` |
| `src` | Resource location (image, video, iframe) | `src="logo.png"` |
| `alt` | Alternate text for an image (screen readers, fallback) | `alt="Company Logo"` |
| `data-*` | Custom data attribute (used for JS or scraping) | `data-id="product-123"` |
| `name` | Name of an input field in forms | `name="email"` |
| `value` | The value of an input field or button | `value="Submit"` |
| `title` | Tooltip text shown on hover | `title="More info"` |

---

#### 📌 Example in Context

```html
<div class="product-card" data-id="123">
  <a href="/product/123" class="product-link">
    <img src="phone.jpg" alt="iPhone 14">
    <span class="price">$999</span>
  </a>
</div>
```

From this, you could extract:
- The **price**: `<span class="price">` → `'$999'`
- The **image source**: `<img src="...">` → `'phone.jpg'`
- The **product link**: `<a href="/product/123">` → `'/product/123'`
- The **custom data-id**: `data-id="123"` → `'123'`

---

---

### 5. 🧰 BeautifulSoup Basics

Web scraping in Python commonly uses two foundational libraries:

#### 🔹 `requests`

- A powerful HTTP client library for Python.
- Used to **send HTTP requests** (like `GET`, `POST`) to websites and receive the response.

#### 🔹 `BeautifulSoup`

- A library for **parsing HTML/XML** documents.
- Allows you to navigate the structure of a webpage and extract elements using tags, classes, attributes, and more.
- Part of the `bs4` package (install with `pip install beautifulsoup4`).

---

#### 🌐 Load a Webpage

```python
from bs4 import BeautifulSoup
import requests

url = "https://example.com"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
```

---

#### 🔍 What Are These Objects?

| Variable | Type | Description |
|----------|------|-------------|
| `response` | `requests.models.Response` | The HTTP response object from `requests.get()` |
| `soup` | `bs4.BeautifulSoup` | The parsed HTML DOM tree |

---

#### ✅ `requests.get()`

This sends an HTTP **GET** request to retrieve the page.

**Common properties and methods of `response`:**

| Attribute/Method | Description |
|------------------|-------------|
| `response.text` | HTML content of the page (as a string) |
| `response.content` | Raw bytes (useful for images, PDFs) |
| `response.status_code` | HTTP status code (200 = OK, 404 = Not Found) |
| `response.headers` | HTTP headers returned by the server |
| `response.json()` | Parses response as JSON (if applicable) |

---

#### ✅ `BeautifulSoup(response.text, 'html.parser')`

This parses the raw HTML and gives you a navigable DOM tree.

**Common methods and patterns with `soup`:**

| Method | Description |
|--------|-------------|
| `soup.find('tag')` | Finds first occurrence of a tag |
| `soup.find_all('tag')` | Finds all matching tags |
| `soup.select('css-selector')` | Selects elements using CSS selectors |
| `soup.get_text()` | Extracts all text from soup or tag |
| `soup.title` | Accesses `<title>` element |
| `soup.prettify()` | Returns nicely indented HTML |

---

#### 📌 Summary

```python
# Step-by-step breakdown
response = requests.get(url)                     # GET request to fetch page
html_content = response.text                     # Raw HTML string
soup = BeautifulSoup(html_content, 'html.parser')# Parsed DOM object
```

You now have full control to search, navigate, and extract data from the page structure using `.find()`, `.select()`, and other tools.

---


---

### 6. 🎯 Selecting Elements

When scraping HTML with BeautifulSoup, there are two main ways to locate elements on a page:

---

### #🧭 Option 1: Basic `.find()` / `.find_all()` Methods

These methods allow you to find elements by tag name, class, id, or attribute.

- `soup.find()` returns the **first matching element**
- `soup.find_all()` returns a **list of all matching elements**
- You can filter further using optional arguments like `class_`, `id`, or `attrs`

| Code | Description | What It Returns |
|------|-------------|------------------|
| `soup.find('tag')` | First occurrence of tag | A BeautifulSoup `Tag` object |
| `soup.find_all('tag')` | All matching tags | A list of `Tag` objects |
| `soup.find('tag', class_='classname')` | Tag with matching class | First matching `Tag` |
| `soup.find('tag')['attribute']` | Get an attribute like `href` | String value of the attribute |

**Example:**
```python
soup.find('div', class_='product')         # First product block
soup.find('a')['href']                    # Href link from first <a>
soup.find_all('span', class_='price')     # List of all price <span>s
```

---

#### 🎨 Option 2: CSS Selectors with `.select()` and `.select_one()`

These use **CSS syntax** — similar to what you'd use in a browser's Developer Tools — and are more powerful when dealing with **deeply nested** or **complex** structures.

| Selector | Meaning |
|----------|---------|
| `.class` | Selects any element with this class |
| `#id` | Selects element by ID |
| `tag[attr="value"]` | Tag with specific attribute |
| `div > span` | Direct child span of a div |
| `div span` | Any descendant span of a div |

**Examples:**
```python
soup.select('div.product-card span.price')        # All price spans inside product cards
soup.select_one('a.buy-link')['href']            # First buy button link
soup.select('table tr td:nth-child(2)')          # Second column of all table rows
```

- `soup.select()` → returns a list of matching tags
- `soup.select_one()` → returns the first match (like `.find()`)

---

#### 🧠 When to Use What?

| Use Case | Best Method |
|----------|-------------|
| Simple tag with known name | `.find()`, `.find_all()` |
| You need all matches | `.find_all()` |
| You know the CSS path | `.select()` or `.select_one()` |
| Deeply nested selectors | `.select()` gives more control |
| Precise control (e.g. nth-child) | Use `.select()` |

---

```python
# Using .find()
div = soup.find('div', class_='product-card')

# Using CSS selector
div = soup.select_one('div.product-card')

# Both are valid – pick what matches your use case
```

---

### 7. 🧪 Example: Scraping a Product Listing

```python
products = soup.find_all('div', class_='product-card')

for p in products:
    name = p.find('h2', class_='product-name').text.strip()
    price = p.find('span', class_='price').text.strip()
    rating = p.find('span', class_='rating').text.strip()
    link = p.find('a', class_='buy-link')['href']
    print(name, price, rating, link)
```

---

### 8. 🧼 Text Extraction & Cleaning

When scraping data from HTML, you often need to **clean up whitespace**, **normalize characters**, and **handle missing tags** gracefully.

| Task | Code | Input | Output |
|------|------|-------|--------|
| Clean whitespace | `element.text.strip()` | `<span> $9.99 </span>` | `'${9.99}'` |
| Raw text fragments | `list(element.strings)` | `<div>Price: <span>$9.99</span></div>` | `['Price: ', '$9.99']` |
| Clean fragments | `list(element.stripped_strings)` | `<div>  Price:  <span>$9.99</span> </div>` | `['Price:', '$9.99']` |
| Normalize unicode | `unicodedata.normalize("NFKD", s)` | `'1 000 kg'` (contains non-breaking spaces) | `'1000 kg'` |
| Handle missing tag safely | `element.find('tag') or ''` | If `element` doesn't contain `tag` | `''` (returns blank instead of error) |

**Tip:** Use `.stripped_strings` when the tag has nested elements but you want a clean list of text fragments.

---

---

### 9. 📊 Scraping Tables Effectively

Tables are commonly used on websites to display structured information such as financial data, product comparisons, or sports stats. To scrape tables effectively:

- You need to **understand the raw HTML structure** (especially how headers, rows, and cells are organized)
- Account for irregularities like:
  - Extra header rows
  - `colspan` or `rowspan` attributes
  - Empty or malformed cells

---

#### 🔹 Basic Table Example

```html
<table>
  <tr><th>Item</th><th>Price</th></tr>
  <tr><td>Phone</td><td>$699</td></tr>
  <tr><td>Laptop</td><td>$999</td></tr>
</table>
```

```python
rows = soup.find_all('tr')[1:]  # Skip header
for row in rows:
    cols = row.find_all('td')
    item = cols[0].text.strip()
    price = cols[1].text.strip()
    print(item, price)
```

**Output:**
```
Phone $699
Laptop $999
```

---

#### 🔹 Intermediate: Handle Table Headers Dynamically

```html
<table>
  <tr><th>Product</th><th>Price</th><th>Rating</th></tr>
  <tr><td>Phone</td><td>$699</td><td>4.5</td></tr>
</table>
```

```python
headers = [th.text.strip() for th in soup.find('tr').find_all('th')]
print("Headers:", headers)

rows = soup.find_all('tr')[1:]
for row in rows:
    values = [td.text.strip() for td in row.find_all('td')]
    data = dict(zip(headers, values))
    print(data)
```

**Output:**
```python
{'Product': 'Phone', 'Price': '$699', 'Rating': '4.5'}
```

---

#### 🔹 Advanced: Tables with `colspan`

```html
<table>
  <tr>
    <th rowspan="2">Item</th>
    <th colspan="2">Price</th>
  </tr>
  <tr>
    <th>USD</th>
    <th>SGD</th>
  </tr>
  <tr>
    <td>Phone</td><td>$699</td><td>S$950</td>
  </tr>
</table>
```

##### 🧠 What’s happening:
- First row defines merged headers: "Price" spans 2 columns.
- Second row defines subheaders: "USD", "SGD"
- Data rows follow.

```python
import pandas as pd

rows = soup.find_all('tr')
table_data = []

for row in rows:
    cells = row.find_all(['td', 'th'])
    row_data = [cell.text.strip() for cell in cells]
    table_data.append(row_data)

df = pd.DataFrame(table_data)
print(df)
```

**Output DataFrame:**
```
       0     1      2
0   Item Price  Price
1         USD    SGD
2  Phone  $699  S$950
```

> ⚠️ You may need to merge headers manually if the table is complex.

---

### 🛠️ Pro Tip: Using `pandas.read_html()` for Simpler Tables

If you're lucky and the table is cleanly structured, you can let pandas do all the work:

```python
import pandas as pd

tables = pd.read_html("https://example.com/page-with-table")
df = tables[0]
print(df.head())
```

- It uses `lxml` or `html5lib` under the hood
- Automatically detects headers and spans
- Perfect for Wikipedia, news sites, and finance sites

---

### 📌 Key Considerations for Table Scraping

| Challenge | What to Do |
|----------|------------|
| Multiple header rows | Use nested parsing logic or pandas to flatten |
| `colspan`/`rowspan` | Adjust alignment manually or pre-clean with pandas |
| Missing values | Use `.get_text(strip=True) or ''` to avoid errors |
| Hidden content (JS) | Use Selenium or scrape an underlying API if available |

---


---

### 10. 📚 Handling Pagination

#### HTML Example

```html
<a href="/products?page=2" class="next">Next</a>
```

#### Python

```python
page = 1
while True:
    url = f"https://example.com/products?page={page}"
    soup = BeautifulSoup(requests.get(url).text, 'html.parser')

    # extract logic...

    next_btn = soup.find('a', class_='next')
    if not next_btn:
        break
    page += 1
```

---

### 11. 📤 Export to CSV

```python
import pandas as pd

data = [
    {"name": "iPhone", "price": "$999", "link": "/iphone"},
    {"name": "Samsung", "price": "$899", "link": "/samsung"}
]

df = pd.DataFrame(data)
df.to_csv("products.csv", index=False)
```

---

### 12. ⚠️ Gotchas & Best Practices

| Issue | Tip |
|-------|-----|
| Site is JavaScript-rendered | Use Selenium or Playwright |
| IP gets blocked | Add `time.sleep()`, rotate proxies or headers |
| Pages return empty | Check `robots.txt` or network tab in browser |
| Data is in API | Use `requests` on API instead of scraping HTML |

---

### ✅ Final Checklist

- [x] Explore with browser **Inspect Element**
- [x] Use `.find()` or `.select()` to locate elements
- [x] Extract `.text` or `['href']`/`['src']` as needed
- [x] Handle nested elements and edge cases
- [x] Export cleaned data into CSV/JSON

---

## 🧪 Practice Targets

| Site | Type |
|------|------|
| [Books to Scrape](http://books.toscrape.com) | Product list |
| [Quotes to Scrape](http://quotes.toscrape.com) | Nested authors & tags |
| [Wikipedia Tables](https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)) | Complex tables |

---

> 🚨 Always respect a site's `robots.txt` file and avoid scraping against terms of use. Be polite: add delays, avoid hammering servers, and cache data when possible.
